# Day 11 — The LLM interpretation layer: making the model explain itself, safely
### Real Estate Machine · Graduation Project

Day 10 produced a model that values a house to within about 10.5% and a SHAP decomposition that says
*why*. Both outputs are numbers. A client wants a paragraph.

That is the entire job of today: turn the numbers into sentences. It is also the part of the project
most likely to be attacked in a defense, because "we added AI" is exactly the kind of claim examiners
have learned to distrust. So the notebook is organised around the attack:

> *"How do you know the language model is not making things up?"*

The answer has to be structural, not hopeful. Four defences, built in this order:

1. **The language model is given every number it is allowed to use, and computes none of them.**
   Price, range, drivers, warnings — all computed in Python, from the Day 10 model, before any prompt
   exists.
2. **Deterministic rules decide when a valuation is suspicious**, not the LLM. A rule fires every
   time; a model notices sometimes.
3. **The output is checked before it is shown.** Every number in the generated text is extracted and
   matched against the evidence. An unmatched number means the paragraph is discarded.
4. **There is always a fallback.** No key, no internet, quota exhausted, provider down, check failed —
   the app still produces a written explanation. Your demo cannot depend on someone else's server.

---

## What you will produce

| Output | Why it exists |
|---|---|
| `app/explain.py` | the evidence layer: one house to a fully-computed dossier |
| `app/llm_explain.py` | the language layer: prompt, API call, grounding check, fallback |
| `Models/reference_stats.pkl` | the training-set facts the sanity checks compare against |
| `reports/llm_cache.json` | cached explanations, so the defense demo runs offline |
| `.env` (never committed) | your API key |

**The sentence that frames all of it, and belongs in your slides:**

> The explanation is computed by SHAP. The language model only writes the sentences, it is given the
> numbers it is allowed to use, and its output is checked against them before anyone sees it.

---
## 1. Setup

Both modules live in `app/` for the same reason `preprocessing.py` does: the Day 12 Streamlit app
imports them. Nothing important is defined inside this notebook, because a function defined in a
notebook cannot be imported by an app.

In [1]:
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA = Path("..") / "Data"
MODELS = Path("..") / "Models"
REPORTS = Path("..") / "reports"
REPORTS.mkdir(exist_ok=True)

sys.path.append(str((Path("..") / "app").resolve()))
import preprocessing as P

raw = pd.read_csv(DATA / "data_clean.csv", dtype={"zipcode": str})
df = P.engineer_features(raw)
split = pd.read_csv(DATA / "split_indices.csv", index_col="row")["split"]
is_train = (split.values == "train")

train_rows = df[is_train]
test_rows = df[~is_train]

cfg = joblib.load(MODELS / "regressor_config.pkl")
print(f"Day 10 model: {cfg['model_name']}   MedAPE {cfg['test_MedAPE_%']:.1f}%   "
      f"PPE10 {cfg['test_PPE10_%']:.0f}%")
print(f"Train {len(train_rows):,}   Test {len(test_rows):,}")

Day 10 model: GB (tuned)   MedAPE 10.5%   PPE10 48%
Train 3,476   Test 869


---
## 2. Reference statistics — what "suspicious" means, defined numerically

A warning like *"this house is unusually large"* needs a threshold, and that threshold has to come
from the **training** data. Not the full dataset: the model's blind spots are defined by what it was
trained on, and a house is only "unseen" relative to the rows the model actually learned from.

Five facts, and each one exists because a specific failure can happen without it:

| Fact | The failure it catches |
|---|---|
| the set of zip codes seen in training | an unknown zip silently becomes the global average — and location is the largest driver in the model, so the valuation is close to meaningless |
| the set of cities seen in training | same, one level coarser |
| the 99th percentile of living area | trees cannot extrapolate, so a huge house is valued as if it were merely large |
| the 10th and 90th percentiles of price | Day 10 measured a systematic bias at both ends: optimistic at the bottom, conservative at the top |
| the median price per square foot in each zip code | a valuation wildly out of line with its own neighbourhood is worth flagging whatever the reason |

Saved as a file, so the app and this notebook read the same thresholds and nobody re-derives one by
hand.

In [2]:
ref = {
    "known_zipcodes": sorted(train_rows["zipcode"].astype(str).unique()),
    "known_cities": sorted(train_rows["city"].astype(str).unique()),
    "sqft_q99": float(train_rows["sqft_living"].quantile(0.99)),
    "price_q10": float(train_rows["price"].quantile(0.10)),
    "price_q90": float(train_rows["price"].quantile(0.90)),
    "zip_ppsf_median": (train_rows["price"] / train_rows["sqft_living"])
                       .groupby(train_rows["zipcode"].astype(str)).median().round(1).to_dict(),
    "built_from_rows": int(len(train_rows)),
}
joblib.dump(ref, MODELS / "reference_stats.pkl")

print(f"{len(ref['known_zipcodes'])} zip codes and {len(ref['known_cities'])} cities seen in training")
print(f"99th percentile of living area : {ref['sqft_q99']:,.0f} sqft")
print(f"price 10th / 90th percentile   : ${ref['price_q10']:,.0f} / ${ref['price_q90']:,.0f}")
print("Saved Models/reference_stats.pkl")

77 zip codes and 43 cities seen in training
99th percentile of living area : 5,002 sqft
price 10th / 90th percentile   : $249,000 / $908,750
Saved Models/reference_stats.pkl


---
## 3. The evidence layer — `app/explain.py`

`HouseValuer.evidence(house)` is the only function the app calls to value a house. It returns a
dictionary containing everything a human or a language model could need, and nothing else:

- the prediction, and the 10th-to-90th-percentile range Day 10 measured
- the market segment from Day 7, and that segment's own typical error
- the top SHAP drivers, converted from log space into percentage effects
- the warnings that fired

Three design decisions worth being able to defend:

**It merges `sqft_living` and `log_sqft_living` back together.** Day 6 kept both, so the model splits
the size effect between two columns. Showing a client "living area lowered the price by 6.3%" *and*
"living area on a log scale lowered it by 2.9%" is two half-truths that look like two separate
reasons. SHAP values are additive, so they can be added back into one honest figure. This is the kind
of small decision that separates a demo from a product.

**It refuses bad input rather than valuing it.** See section 4.

**It never returns a bare number.** Every dossier carries the range and the model's measured error,
so there is no code path in the app that can display a price without its uncertainty.

In [3]:
from explain import HouseValuer

valuer = HouseValuer(MODELS)

house = {
    "bedrooms": 3, "bathrooms": 2.0, "sqft_living": 1800, "sqft_lot": 7500,
    "floors": 1.0, "waterfront": 0, "view": 0, "condition": 3,
    "sqft_basement": 500, "yr_built": 1985,
    "city": "Seattle", "zipcode": "98115",
}

evidence = valuer.evidence(house)
print(json.dumps(evidence, indent=2, default=str))

{
  "model_name": "GB (tuned)",
  "house": {
    "bedrooms": 3,
    "bathrooms": 2.0,
    "sqft_living": 1800,
    "sqft_lot": 7500,
    "floors": 1.0,
    "waterfront": 0,
    "view": 0,
    "condition": 3,
    "sqft_basement": 500,
    "yr_built": 1985,
    "city": "Seattle",
    "zipcode": "98115"
  },
  "house_age": 29,
  "predicted_price": 564600.0,
  "range_low": 446600.0,
  "range_high": 706600.0,
  "price_per_sqft": 314,
  "typical_error_pct": 10.5,
  "within_10pct_of_sale_price": 48,
  "segment": "Established Family Home",
  "segment_typical_error_pct": 9.7,
  "drivers": [
    {
      "feature": "zipcode_te",
      "label": "the zip code",
      "value": "98115",
      "pct_effect": 21.6
    },
    {
      "feature": "sqft_living",
      "label": "living area",
      "value": "1,800 sqft",
      "pct_effect": -9.0
    },
    {
      "feature": "city_te",
      "label": "the city",
      "value": "Seattle",
      "pct_effect": 7.9
    },
    {
      "feature": "view",
      "la

---
## 4. Garbage in, refusal out

The roadmap asks the Day 12 app to "handle bad input (0 bedrooms, negative sqft)". Handling it does
not mean clamping it to something sensible and carrying on. A house with zero square feet is not an
unusual house — it is a typing mistake, and the only correct response is to refuse.

The rule to state: **a model should decline to answer a question it was never trained to answer.**
Every check below is a range the training data actually covers.

Note also *how* it refuses: all the problems at once, not the first one. A form that reports one error
per submission is a form people give up on.

In [4]:
bad_houses = {
    "zero living area": {**house, "sqft_living": 0},
    "negative basement": {**house, "sqft_basement": -50},
    "basement larger than the house": {**house, "sqft_basement": 5000},
    "impossible year": {**house, "yr_built": 2027},
    "several problems at once": {**house, "sqft_living": 0, "bedrooms": 99, "condition": 9},
    "missing a field": {k: v for k, v in house.items() if k != "zipcode"},
}

for label, bad in bad_houses.items():
    try:
        valuer.evidence(bad)
        print(f"{label:<32} ACCEPTED  <- this is a bug, the check is missing")
    except ValueError as exc:
        print(f"{label:<32} refused: {exc}")

zero living area                 refused: living area must be greater than zero; basement area cannot exceed total living area
negative basement                refused: basement area cannot be negative
basement larger than the house   refused: basement area cannot exceed total living area
impossible year                  refused: year built must be between 1850 and 2014
several problems at once         refused: living area must be greater than zero; bedrooms must be between 0 and 15; condition rating must be between 1 and 5; basement area cannot exceed total living area
missing a field                  refused: missing required fields: zipcode


---
## 5. The sanity checks — rules, not vibes

The roadmap asks the LLM to *"flag if it looks unrealistic"*. **Do not do that.** A language model
asked to judge plausibility will be inconsistent between runs, cannot tell you why it decided, and has
no access to the training distribution. It will also miss the case that matters most — an unknown zip
code — because nothing in the text looks wrong.

So the flags are computed in Python, and the LLM is *told* which fired. Its job is to phrase them.

Four houses, each designed to trip a different rule.

In [5]:
scenarios = {
    "ordinary Seattle house": house,
    "cheapest end of the market": {
        "bedrooms": 2, "bathrooms": 1.0, "sqft_living": 860, "sqft_lot": 6000,
        "floors": 1.0, "waterfront": 0, "view": 0, "condition": 3,
        "sqft_basement": 0, "yr_built": 1955, "city": "Auburn", "zipcode": "98002"},
    "waterfront luxury": {
        "bedrooms": 5, "bathrooms": 3.5, "sqft_living": 4200, "sqft_lot": 12000,
        "floors": 2.0, "waterfront": 1, "view": 4, "condition": 4,
        "sqft_basement": 800, "yr_built": 2005, "city": "Bellevue", "zipcode": "98004"},
    "a city the model has never seen": {
        "bedrooms": 3, "bathrooms": 2.0, "sqft_living": 1800, "sqft_lot": 7500,
        "floors": 1.0, "waterfront": 0, "view": 0, "condition": 3,
        "sqft_basement": 0, "yr_built": 1990, "city": "Portland", "zipcode": "97201"},
}

cases = {}
for label, h in scenarios.items():
    ev = cases[label] = valuer.evidence(h)
    print(f"\n{'=' * 78}\n{label}: ${ev['predicted_price']:,.0f}  "
          f"(range ${ev['range_low']:,.0f} - ${ev['range_high']:,.0f})   segment: {ev['segment']}")
    for d in ev["drivers"][:3]:
        print(f"    {d['label']} = {d['value']}: {d['pct_effect']:+.1f}%")
    for f in ev["flags"]:
        print(f"    FLAG  {f}")
    if not ev["flags"]:
        print("    no flags")


ordinary Seattle house: $564,600  (range $446,600 - $706,600)   segment: Established Family Home
    the zip code = 98115: +21.6%
    living area = 1,800 sqft: -9.0%
    the city = Seattle: +7.9%
    no flags

cheapest end of the market: $172,700  (range $136,600 - $216,100)   segment: Compact Entry-Level
    the zip code = 98002: -35.0%
    living area = 860 sqft: -28.4%
    the city = Auburn: -12.0%
    FLAG  This valuation is in the cheapest tenth of the market, where the model is known to be optimistic by roughly 12%. Treat it as an upper bound.

waterfront luxury: $3,471,300  (range $2,746,100 - $4,344,700)   segment: Premium View Property
    the zip code = 98004: +87.5%
    living area = 4,200 sqft: +74.4%
    view rating = 4 out of 4: +38.8%
    FLAG  Waterfront property: the training data contains very few of these, so this valuation rests on almost no comparable sales.
    FLAG  This valuation is in the most expensive tenth of the market, where the model is systematically co

### What each case proves

**The ordinary house** trips nothing. That matters: a checker that fires on everything is ignored,
exactly like a smoke alarm that goes off when you make toast.

**The cheap house** trips the low-end warning, and the wording of that warning comes straight out of
Day 10's decile analysis — the model over-values the bottom tenth of the market by roughly 12%, so the
client is told to treat the figure as an upper bound. **The warning is a measured number from your own
notebook, not a hedge.**

**The waterfront luxury house** trips four rules at once, and this is the most important case in the
notebook. The model returns a confident multi-million-dollar figure, and every one of those four flags
says do not trust it: almost no waterfront training data, the conservative top decile, a price per
square foot far above its zip code's median, and the segment Day 10 identified as twice as inaccurate
as the others. **A model that says nothing when it is out of its depth is more dangerous than one that
is merely inaccurate.**

**The Portland house** is the case a language model would never catch on its own. The text reads
perfectly normally; the failure is invisible in the words. The zip code is not in the training set, so
the largest single driver in the model has silently collapsed to the market-wide average, and the
valuation is close to meaningless. Only a rule with access to the training data can know that.

> **Defense answer:** the language model does not decide whether a valuation is trustworthy. Four
> deterministic rules do, using thresholds computed from the training set, and the language model is
> told which ones fired.

---
## 6. The prompt

Every rule below exists because of a specific failure mode. Read the printed prompt and match each
line to its rule.

| Prompt element | The failure it prevents |
|---|---|
| "Use ONLY the figures given" | the classic hallucinated statistic — a market average, a mortgage rate, a comparable sale |
| "You are a writer, not an appraiser" | the model volunteering a valuation opinion of its own |
| "Never claim the valuation is accurate" | confident prose making a 10.5%-error model sound like a survey |
| "warnings are the most important part" | the caveats being buried under fluent, reassuring sentences |
| a fixed word count and structure | unpredictable output length breaking the app's layout |
| "no markdown, no greeting" | asterisks and "Dear customer" rendering as literal text in Streamlit |
| flat labelled evidence, not JSON | JSON in a prompt invites the model to reply in JSON, and is harder to show an examiner |
| temperature 0.2 | near-deterministic output; you are formatting facts, not writing poetry |

Notice what is *not* in the prompt: the raw feature vector, the model type's internals, anything about
zip codes beyond the number, and any instruction to be positive or reassuring. The model cannot leak
what it was never given.

In [6]:
from llm_explain import build_prompt

prompt = build_prompt(evidence)
print(prompt)
print(f"\n[prompt length: {len(prompt)} characters, roughly {len(prompt) // 4} tokens]")

You explain house valuations produced by a statistical model to a
non-technical client. You are a writer, not an appraiser.

Rules, all of them absolute:
1. Use ONLY the figures given in the evidence below. Never introduce a number that is
   not there - no market averages, no interest rates, no dates, no comparable sales,
   no percentages you worked out yourself.
2. Never claim the valuation is accurate. Report the stated typical error and the
   stated range as what they are: this model's measured performance on houses it had
   not seen.
3. If warnings are listed, they are the most important part of your answer. State each
   one plainly in your own words. Do not soften them and do not skip one.
4. Do not speculate about why a feature matters, beyond what the evidence says.
5. No greeting, no sign-off, no bullet points, no headings, no markdown.

EVIDENCE
--------
Predicted price: $564,600
Honest range (8 out of 10 houses fall in it): $446,600 to $706,600
Implied price per square f

---
## 7. Calling the provider — and the four ways it fails

Two free options, both fine for this project: **Google Gemini** and **Groq**. Put whichever key you
have in a `.env` file in the project root:

```
GEMINI_API_KEY=...
# or
GROQ_API_KEY=...
```

`.gitignore` already excludes `.env`. **Check that before your first push** — a key committed to a
public GitHub repository is scraped within minutes, and it is the single most common way a student
project leaks a credential.

**Model names go stale.** The name in `llm_explain.py` is a default that can be overridden in `.env`
with `GEMINI_MODEL=` or `GROQ_MODEL=`. Before the defense, run

```powershell
python app/llm_explain.py --list-models
```

which asks the provider what your key can actually use. Do not trust a model name from a tutorial,
including this one.

**Four failures, all handled, none of which should break a live demo:**

1. **No key configured** → fall back, and say why in the returned `error` field.
2. **Network or provider error** (timeout, quota, 500, retired model name) → caught, fall back.
3. **A slow response** → a 20-second timeout, so the app cannot hang forever on a stranger's server.
4. **A fluent lie** → the grounding check in section 8 rejects it, fall back.

The function never raises and always returns text. That is a hard requirement for something on a
demo path.

In [7]:
from llm_explain import available_provider, explain_prediction

print("Provider detected from .env:", available_provider() or "none")

t0 = time.time()
result = explain_prediction(evidence, use_cache=False)
elapsed = time.time() - t0

print(f"source   : {result['source']}")
print(f"grounded : {result['grounded']}")
print(f"error    : {result['error']}")
print(f"latency  : {elapsed:.2f}s\n")
print(result["text"])

Provider detected from .env: groq


source   : groq
grounded : True
error    : None
latency  : 1.36s

The model values your home at $564,600. In eight out of ten cases, the true price falls between $446,600 and $706,600. This implies a price of $314 per square foot for your 1,800 square foot property.

Three factors most influenced this figure. Living in zip code 98115 raised the value by 21.6%, while being in Seattle added 7.9%. Conversely, your living area of 1,800 square feet lowered the valuation by 9.0%. These specific features drove the final number more than other aspects of the house.

You should treat this figure with caution. On unseen houses, this model’s typical error is 10.5%, meaning it is often off by that amount. Only 48% of unseen houses were valued within 10% of their actual sale price. For your specific market segment, the typical error is 9.7%. No warnings were triggered for this house, but the model does not guarantee accuracy. It is a statistical estimate, not a precise appraisal.


If the output above says `source: fallback` and `error: no API key configured`, that is the
no-key path working correctly — not a bug. Add a key and re-run this cell to see the language model
version.

**Run this cell both ways before the defense**, and keep the fallback output where you can find it.
"Here is what it produces when the API is down" is a much better answer than discovering it live.

---
## 8. The grounding check — the part that makes this defensible

This is the piece that turns "we told it not to make things up" into an engineering claim.

`check_grounding(text, evidence)` extracts every number from the generated paragraph, and compares
each one against the list of values the evidence contains. A 2% tolerance lets the model round
\$564,600 to \$565,000 without being punished for it; anything outside the tolerance is a number the
model invented, and the paragraph is discarded in favour of the fallback.

It is deliberately a **whitelist**, not a blacklist. You cannot enumerate the lies a language model
might tell. You can enumerate the numbers it is allowed to say.

The test below is the one that matters: a paragraph that is entirely reasonable, fluent, on-topic —
and contains one invented comparable sale. This is exactly what a real hallucination looks like, and
it is why "the output read fine to me" is not a quality check.

In [8]:
from llm_explain import check_grounding, fallback_explanation

honest = fallback_explanation(evidence)

hallucinated = (
    "The model values this house at $564,600, which is broadly in line with the "
    "Seattle market. Comparable three-bedroom homes in 98115 sold for an average of "
    "$612,400 last quarter, and prices in the area have risen 4.8% year on year. "
    "You can be confident this is an accurate valuation."
)

rounded = (
    "The model values this house at about $565,000, with a likely range of roughly "
    "$447,000 to $707,000. Its typical error on unseen houses is around 10%."
)

for label, text in [("evidence-only text", honest),
                    ("fluent but invented", hallucinated),
                    ("rounded, still honest", rounded)]:
    ok, bad = check_grounding(text, evidence)
    verdict = "PASS" if ok else "REJECTED"
    print(f"{label:<24} {verdict:<9} ungrounded numbers: {bad}")

evidence-only text       PASS      ungrounded numbers: []
fluent but invented      REJECTED  ungrounded numbers: [612400.0]
rounded, still honest    PASS      ungrounded numbers: []


### Read the three verdicts

**The evidence-only text passes**, as it must — every figure came from the dossier.

**The fluent invention is rejected**, and look at what it invented: a comparable sale price and a
year-on-year growth rate. Neither is in the evidence, both sound completely plausible, and a reader
skimming the paragraph would have no way to tell. The check catches them because it is not reading
for plausibility, it is checking membership in a list.

That paragraph also breaks a second rule — *"you can be confident this is an accurate valuation"* —
which the numeric check cannot catch. **Say this limitation out loud before an examiner finds it:**
the grounding check verifies numbers, not claims. It is a strong guard against fabricated statistics
and no guard at all against overconfident phrasing. That is what the prompt rules and the low
temperature are for, and they are weaker guarantees.

**The rounded version passes**, which is the tolerance doing its job. A check so strict that it
rejects "about $565,000" would fire constantly and be switched off within a week — and a safety check
that gets switched off protects nothing.

---
## 9. The fallback — a real deliverable, not a placeholder

`fallback_explanation` assembles a paragraph from the same evidence with no API at all. It runs when
there is no key, when the network is down, when the quota is gone, and when the grounding check
rejects the model's output.

It has to read well enough to put in front of a client, because on demo day it might be what a client
sees. Read the three below and judge them on that standard.

The honest comparison to draw in your defense: the fallback is stiffer and more repetitive than a
language model's prose, and it is **exactly as correct**. What the LLM adds is fluency and variation,
not information — every fact in both versions comes from the same dossier. That is a precise
statement of what the AI layer contributes, and it is much better than implying the AI is doing the
thinking.

In [9]:
for label, ev in list(cases.items())[:3]:
    print("=" * 78)
    print(label.upper())
    print("=" * 78)
    print(fallback_explanation(ev))
    print()

ORDINARY SEATTLE HOUSE
The model values this 3-bedroom, 1,800 sqft house in Seattle (98115) at $564,600, or about $314 per square foot. Eight out of ten houses like it fall between $446,600 and $706,600, and that range is the figure to work with rather than the single number.

What pushed the valuation up: the zip code (98115), raising it by about 21.6% and the city (Seattle), raising it by about 7.9%. What pulled it down: living area (1,800 sqft), lowering it by about 9.0% and view rating (0 out of 4), lowering it by about 1.9%. This house falls into the Established Family Home segment of the market.

On houses it had never seen, this model was typically 10.5% away from the actual sale price, and it landed within 10% of the sale price 48% of the time. It is a screening tool, not a formal valuation. For the Established Family Home segment specifically, the typical error is 9.7%.

CHEAPEST END OF THE MARKET
The model values this 2-bedroom, 860 sqft house in Auburn (98002) at $172,700, o

---
## 10. Caching — how the demo survives no internet

The roadmap's risk table says it plainly: *"LLM API fails on demo day → cache one example response."*

`explain_prediction` caches by the house's input values, in `reports/llm_cache.json`. Warm the cache
now with the houses you intend to demo, commit the file, and the demo runs with the WiFi off.

Two side benefits worth mentioning: repeated identical requests cost nothing and return instantly,
and the cached text is reproducible — the same house shows the same paragraph every time, which is
what you want when a panel asks you to run it twice.

In [10]:
warmed = {}
for label, ev in cases.items():
    res = explain_prediction(ev, use_cache=True)
    warmed[label] = res
    print(f"{label:<34} source={res['source']:<9} grounded={res['grounded']}")

cache_path = REPORTS / "llm_cache.json"
print(f"\nCache file: {cache_path}  exists={cache_path.exists()}")
if cache_path.exists():
    print(f"{len(json.loads(cache_path.read_text()))} entries cached")
else:
    print("Nothing cached - the fallback path does not write to the cache, by design:")
    print("caching a fallback would hide a broken API behind a stale-looking success.")

ordinary Seattle house             source=groq      grounded=True


cheapest end of the market         source=groq      grounded=True


waterfront luxury                  source=groq      grounded=True


a city the model has never seen    source=fallback  grounded=True

Cache file: ..\reports\llm_cache.json  exists=True
3 entries cached


---
## 11. End to end on real houses, including one the model gets wrong

Everything so far used invented houses. Now run three **test-set** houses — rows the model has never
seen — where the true sale price is known, and put the explanation next to the truth.

Choosing them honestly matters. Picking three houses the model nailed would be a demo, not a test. So:
one typical case, one the model badly over-values, and one from the Premium View Property segment that
Day 10 identified as its weakest.

In [11]:
model = joblib.load(MODELS / "regressor.pkl")
pred_test = np.expm1(model.predict(test_rows[cfg["features"]]))
actual_test = test_rows["price"].values
signed = (pred_test - actual_test) / actual_test

segments = pd.read_csv(DATA / "data_clustered.csv")["segment"].values[~is_train]

# "Typical" means typical in two senses: an ordinary house from the middle of the
# market, with an error close to the model's median. Taking the median error alone
# would be honest but could land on a mansion.
middle = ((actual_test >= np.percentile(actual_test, 25)) &
          (actual_test <= np.percentile(actual_test, 75)))
median_abs_err = np.median(np.abs(signed))
typical = int(np.argmin(np.where(middle, np.abs(np.abs(signed) - median_abs_err), np.inf)))

picks = {
    "typical (median error)": typical,
    "badly over-valued": int(np.argmax(signed)),
    "premium segment": int(np.where(segments == "Premium View Property")[0][0]),
}


def to_house(row):
    return {
        "bedrooms": float(row["bedrooms"]), "bathrooms": float(row["bathrooms"]),
        "sqft_living": int(row["sqft_living"]), "sqft_lot": int(row["sqft_lot"]),
        "floors": float(row["floors"]), "waterfront": int(row["waterfront"]),
        "view": int(row["view"]), "condition": int(row["condition"]),
        "sqft_basement": int(row["sqft_basement"]), "yr_built": int(row["yr_built"]),
        "yr_renovated": row["yr_renovated"], "city": row["city"], "zipcode": str(row["zipcode"]),
    }


real_cases = {}
for label, i in picks.items():
    row = test_rows.iloc[i]
    ev = valuer.evidence(to_house(row))
    real_cases[label] = (ev, float(actual_test[i]))
    inside = ev["range_low"] <= actual_test[i] <= ev["range_high"]
    print(f"{label:<24} actual ${actual_test[i]:>10,.0f}   predicted ${ev['predicted_price']:>10,.0f}"
          f"   ({100 * signed[i]:+6.1f}%)   true price inside the quoted range: {inside}")

typical (median error)   actual $   490,000   predicted $   541,700   ( +10.5%)   true price inside the quoted range: True
badly over-valued        actual $   225,000   predicted $   513,600   (+128.2%)   true price inside the quoted range: False


premium segment          actual $   491,500   predicted $   406,200   ( -17.4%)   true price inside the quoted range: True


In [12]:
for label, (ev, actual) in real_cases.items():
    res = explain_prediction(ev, use_cache=True)
    print("=" * 78)
    print(f"{label.upper()}   |   actual sale price ${actual:,.0f}   |   source: {res['source']}")
    print("=" * 78)
    print(res["text"])
    print()

TYPICAL (MEDIAN ERROR)   |   actual sale price $490,000   |   source: fallback
The model values this 2-bedroom, 1,840 sqft house in Seattle (98107) at $541,700, or about $294 per square foot. Eight out of ten houses like it fall between $428,500 and $678,000, and that range is the figure to work with rather than the single number.

What pushed the valuation up: the zip code (98107), raising it by about 19.8% and the city (Seattle), raising it by about 12.6%. What pulled it down: living area (1,840 sqft), lowering it by about 5.5% and share of the space that is below grade (39% of the floor area), lowering it by about 4.3%. This house falls into the Compact Entry-Level segment of the market.

On houses it had never seen, this model was typically 10.5% away from the actual sale price, and it landed within 10% of the sale price 48% of the time. It is a screening tool, not a formal valuation. For the Compact Entry-Level segment specifically, the typical error is 10.9%.



BADLY OVER-VALUED   |   actual sale price $225,000   |   source: fallback
The model values this 2-bedroom, 1,396 sqft house in Redmond (98053) at $513,600, or about $368 per square foot. Eight out of ten houses like it fall between $406,300 and $642,800, and that range is the figure to work with rather than the single number.

What pushed the valuation up: the zip code (98053), raising it by about 19.3% and lot size (111,949 sqft), raising it by about 14.3%. What pulled it down: living area (1,396 sqft), lowering it by about 21.9% and bathrooms (1), lowering it by about 5.6%. This house falls into the Compact Entry-Level segment of the market.

On houses it had never seen, this model was typically 10.5% away from the actual sale price, and it landed within 10% of the sale price 48% of the time. It is a screening tool, not a formal valuation. For the Compact Entry-Level segment specifically, the typical error is 10.9%.



PREMIUM SEGMENT   |   actual sale price $491,500   |   source: fallback
The model values this 4-bedroom, 2,190 sqft house in Auburn (98092) at $406,200, or about $185 per square foot. Eight out of ten houses like it fall between $321,300 and $508,400, and that range is the figure to work with rather than the single number.

What pushed the valuation up: lot size (125,452 sqft), raising it by about 20.0% and view rating (2 out of 4), raising it by about 11.3%. What pulled it down: the zip code (98092), lowering it by about 29.8% and the city (Auburn), lowering it by about 9.0%. This house falls into the Premium View Property segment of the market.

On houses it had never seen, this model was typically 10.5% away from the actual sale price, and it landed within 10% of the sale price 48% of the time. It is a screening tool, not a formal valuation. For the Premium View Property segment specifically, the typical error is 17.9%. Please note: 'Premium View Property' is the segment this model 

### What to notice, especially about the one it got wrong

**The typical case** lands close, and its true price falls inside the quoted range. That is the
result to lead with — and note that it is the *range* that was right, not the point estimate.

**The badly over-valued case** is the one to talk about. The explanation is confident, well written,
and wrong, and nothing in the prose signals it. Look at what the drivers say about it: a large lot in
an expensive zip code, pushing the valuation up hard, on a house that sold for less than half the
prediction. The model has no way to know what is wrong with that property, because whatever it is —
condition, access, a teardown, a family sale — is not in any of the twenty columns. That is not a flaw in the language layer; it is the
model's error surfacing honestly. The right lesson, and the right sentence for the defense:

> The explanation layer explains what the model did. It cannot detect that the model was wrong, and
> nothing in this project claims it can. What it can do is carry the range and the measured error into
> every answer, so a user is never shown a bare number — and this house's true price sits outside the
> quoted range, which is exactly what "8 out of 10" means for the other 2.

**The premium case** carries its segment warning, which is the system working: the model's known weak
spot arrives attached to the valuation instead of buried in a notebook.

Show all three on demo day. A student who volunteers a failure case and explains it is more convincing
than one who shows three successes.

---
## 12. What this layer must never do

Worth a slide, and worth having thought about before you are asked.

**It must never claim to be a valuation.** The output is a screening estimate from a model with a
10.5% typical median error, trained on ten weeks of 2014 sales in one US county. In most jurisdictions
a formal valuation is a regulated activity. The wording throughout says "the model values this at",
never "this house is worth".

**It must never explain the price using anything about the people.** The dataset contains no
demographic information, and none must ever be added to the prompt. Property valuation that varies
with neighbourhood demographics is the mechanism of historic redlining, and it is a live regulatory
issue for automated valuation models. `zipcode` is already a coarse geographic proxy, which is a
limitation you should state proactively rather than defend when asked.

**It must never be presented as a human opinion.** If this were deployed, the paragraph would carry a
line saying it was generated automatically. That is increasingly a legal requirement and it is simply
honest.

**It must never let fluency stand in for accuracy.** The grounding check enforces this for numbers.
For claims, the prompt and the low temperature are the only defence, and they are weaker — which is
why the range and the error figure appear in every single explanation, including the fallback.

> **If an examiner asks "why use an LLM at all, if it only rephrases numbers you already have?"** —
> that is the right question and it has an answer. It converts a technical output into something a
> non-technical client will actually read, which is the difference between a model that is used and a
> model that is ignored. It adds fluency, not information, and the architecture is built so that it
> cannot add anything else.

---
## 13. Deliverables & defense prep

**Produced today**

- `Notebooks/10_llm_explanation.ipynb`
- `app/explain.py` — the evidence layer (SHAP drivers, segment, deterministic checks)
- `app/llm_explain.py` — prompt, providers, grounding check, fallback, cache
- `Models/reference_stats.pkl`
- `reports/llm_cache.json` (once a key is configured and the cache is warmed)

**Before you close the laptop**

1. Create `.env` with a key from Google AI Studio or Groq, and confirm `git status` does not list it.
2. Run `python app/llm_explain.py --list-models` and put a current model name in `.env`.
3. Re-run sections 7 and 11 with the key, read the output, and warm the cache.
4. Save one screenshot of an LLM explanation and one of the fallback. If the WiFi fails during the
   defense, you still have both.

**Commit**

```powershell
git add . && git commit -m "Day 11: LLM interpretation layer with SHAP evidence, grounding check and offline fallback"
```

**Defense questions**

1. *"How do you stop it hallucinating?"* — Four layers. It is given every number it may use and
   computes none of them; it is forbidden from introducing others; every number in its output is
   extracted and matched against the evidence before display, with a 2% rounding tolerance; and
   anything that fails falls back to a deterministic paragraph. I can demonstrate the check rejecting
   a fabricated comparable sale.
2. *"What does the AI actually contribute?"* — Fluency, not information. Every fact in the LLM version
   also appears in my fallback version, because both are generated from the same computed dossier.
   The contribution is that a client reads a paragraph instead of a SHAP chart.
3. *"What if the API is down during this demo?"* — Then you see the fallback, which is written from
   the same evidence, and the responses for my demo houses are cached to disk anyway. The function
   cannot raise and always returns text.
4. *"Who decides whether a valuation looks unrealistic?"* — Deterministic rules with thresholds from
   the training set, not the language model. Unknown zip code, unknown city, living area above the
   99th percentile, price in the top or bottom decile, price per square foot far from the zip code
   median, waterfront, and the weakest segment. The LLM is told which fired and phrases them.
5. *"Where is your API key?"* — In a `.env` file excluded by `.gitignore`, read at runtime with
   `python-dotenv`, and in Streamlit Secrets when deployed on Day 13. It is not in the repository, not
   in a notebook, and not in the model config.
6. *"Could the explanation be discriminatory?"* — The dataset has no demographic fields and none are
   added to the prompt. Zip code is a coarse location proxy, which I state as a limitation. A
   production system would need a fairness review across neighbourhoods before deployment, and my IAAO
   vertical-equity statistic from Day 10 is the closest thing this project has to one.

**Tomorrow (Day 12)** is the Streamlit app, and most of it already exists: `valuer.evidence(house)`
returns the dossier, `explain_prediction(evidence)` returns the paragraph. The app is a form, two
function calls, and a layout. That was the point of putting all of this in `app/` rather than in a
notebook.